In [0]:
def write_user_post_dwell_time_table(spark, run_date, db, table_names): 
    spark.sql("""
              INSERT OVERWRITE TABLE {db}.{user_post_dwell_time} PARTITION (partition_date = '{run_date}')
              with user_set as (
                select distinct customer_id as user_id
                from bz_cn_dl.datalake_community.dw_cust_community_tb_customer
                where coalesce(to_date(from_unixtime(update_at / 1000)),to_date(from_unixtime(create_at / 1000))) <= '{run_date}'
                and delete_type = 0
                and `identity` != 2
                and agree_community_time is not null
              ),
              post_set as (
                select distinct `id` as post_id
                from bz_cn_dl.datalake_community.dw_cust_community_tb_post
                where is_hide = 0
                and audit_status = 1 --审核状态：0-审核中，1-已通过，2-未过审，3-待人工
                and publish_approval_status in (2,3) --发布批准状态：0-待提交，1-待审核，2-已通过，3-已发布，4-已驳回
                and delete_at is null
                and to_date(from_unixtime(post_time/1000)) between date_sub('{run_date}', 365) and '{run_date}' --only collect the latest one year posts
                group by 1
              )
              select user_id, content_id as post_id, sum(coalesce(event_duration,0)) as dwell_time
              from bz_cn_dl.datalake_mystar.dw_cust_dwd_mystar_app_engagement_i_d
              where event_name = 'mystar_page_progress'
              and page_name = 'community_content'
              and user_id in (select user_id from user_set)
              and content_id in (select post_id from post_set)
              and dt = date_sub('{run_date}', 1) --only available after 2024-07-30
              group by 1, 2
              """.format(db = db,
                         user_post_dwell_time = table_names["user_post_dwell_time"],
                         run_date = run_date)
    )

In [0]:
import json

# Get pipeline parameters
run_date = dbutils.widgets.get("run_date")
print("Input parameter run_date:{}".format(run_date))

config_path = dbutils.widgets.get("config_path")
with open(config_path, "r") as file:
    config = json.load(file)
db, table_names = config['DATABASE'], config['TABLE_NAMES']

In [0]:
write_user_post_dwell_time_table(spark, run_date, db, table_names)